In [2]:
import time
import threading
from tqdm.auto import tqdm


class ProgressManager:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._position = 0
        return cls._instance

    def get_pbar(self, iterable, desc, **kwargs):
        with self._lock:
            pos = self._position
            self._position += 1
        pbar = tqdm(iterable, desc=desc, position=pos, leave=(pos == 0), **kwargs)
        return pbar, pos

    def release(self, pos):
        with self._lock:
            self._position -= 1

    def reset(self):
        self._position = 0


# --- Modules with internal nested loops ---


class DetectorModule:
    """Deepest level: pixel rows x pixel cols"""

    def run(self, image):
        pm = ProgressManager()
        pbar, pos = pm.get_pbar(range(4), desc="        🖥️  Detector rows")
        try:
            for row in pbar:
                for col in range(4):  # too fine-grained to track, just compute
                    time.sleep(0.01)
        finally:
            pbar.close()
            pm.release(pos)


class AberrationModule:
    """Applies lens aberrations across Z slices, each slice hits detector"""

    def run(self, volume):
        pm = ProgressManager()
        pbar, pos = pm.get_pbar(range(3), desc="      🔭 Aberration slices")
        try:
            for z_slice in pbar:
                time.sleep(0.05)
                self.detector = DetectorModule()
                self.detector.run(z_slice)
        finally:
            pbar.close()
            pm.release(pos)


class ScatteringModule:
    """Scattering events per atom type, each triggers aberrations"""

    def run(self, volume):
        pm = ProgressManager()
        pbar, pos = pm.get_pbar(range(3), desc="    ⚡ Scattering atoms")
        try:
            for atom in pbar:
                time.sleep(0.05)
                self.aberration = AberrationModule()
                self.aberration.run(atom)
        finally:
            pbar.close()
            pm.release(pos)


class IceModule:
    """Ice slabs, each slab runs scattering independently"""

    def run(self, volume):
        pm = ProgressManager()
        pbar, pos = pm.get_pbar(range(3), desc="  🧊 Ice slabs")
        try:
            for slab in pbar:
                time.sleep(0.05)
                self.scattering = ScatteringModule()
                self.scattering.run(slab)
        finally:
            pbar.close()
            pm.release(pos)


class CrowdingModule:
    """Adds crowder particles — no further nesting, just a flat loop"""

    def run(self, volume):
        pm = ProgressManager()
        pbar, pos = pm.get_pbar(range(5), desc="  👥 Crowders")
        try:
            for crowder in pbar:
                time.sleep(0.05)
        finally:
            pbar.close()
            pm.release(pos)


class Simulator:
    def __init__(self, use_ice=True, use_crowding=True):
        self.use_ice = use_ice
        self.use_crowding = use_crowding

    def run(self, particles):
        pm = ProgressManager()
        pm.reset()
        pbar, pos = pm.get_pbar(particles, desc="🔬 Particles")
        try:
            for particle in pbar:
                if self.use_crowding:
                    CrowdingModule().run(particle)
                if self.use_ice:
                    IceModule().run(particle)
                else:
                    # skip ice, go straight to scattering
                    ScatteringModule().run(particle)
                time.sleep(0.05)
        finally:
            pbar.close()
            pm.release(pos)


# --- Run all combos ---

print("=== Full pipeline: ice + crowding ===")
Simulator(use_ice=True, use_crowding=True).run(range(2))

print("\n=== No ice (scattering runs directly) ===")
Simulator(use_ice=False, use_crowding=True).run(range(2))

print("\n=== No crowding ===")
Simulator(use_ice=True, use_crowding=False).run(range(2))

=== Full pipeline: ice + crowding ===


🔬 Particles:   0%|          | 0/2 [00:00<?, ?it/s]

  👥 Crowders:   0%|          | 0/5 [00:00<?, ?it/s]

  🧊 Ice slabs:   0%|          | 0/3 [00:00<?, ?it/s]

    ⚡ Scattering atoms:   0%|          | 0/3 [00:00<?, ?it/s]

...

        🖥️  Detector rows:   0%|          | 0/4 [00:00<?, ?it/s]

      🔭 Aberration slices:   0%|          | 0/3 [00:00<?, ?it/s]

        🖥️  Detector rows:   0%|          | 0/4 [00:00<?, ?it/s]

        🖥️  Detector rows:   0%|          | 0/4 [00:00<?, ?it/s]

        🖥️  Detector rows:   0%|          | 0/4 [00:00<?, ?it/s]